# Code mẫu — Image segmentation

Segmentation dự đoán nhãn cho từng pixel, nên phải giữ đúng vị trí của từng pixel từ đầu tới cuối. Các ví dụ dưới đây đi từ metric cho vùng vật thể, qua một U-Net nhỏ, tới bước đóng gói mask thành chuỗi RLE để nộp bài.

## Ví dụ 1: Bỏ sót toàn bộ vật thể mà accuracy vẫn cao

Một mask 10×10 có đúng bốn pixel mực, gọi là foreground. Ta so hai dự đoán. Dự đoán thứ nhất đoán toàn bộ là nền. Dự đoán thứ hai tìm được ba trong bốn pixel mực, và đánh dấu nhầm thêm một pixel ở góc. Ta quy ước rằng nếu cả dự đoán lẫn nhãn đều rỗng, Dice và IoU bằng 1.

In [ ]:
import numpy as np

def metrics(prediction, target):
    p, g = np.asarray(prediction, bool), np.asarray(target, bool)
    assert p.shape == g.shape
    intersection = int((p & g).sum())
    union = int((p | g).sum())
    total = int(p.sum() + g.sum())
    return (float((p == g).mean()),
            1.0 if union == 0 else intersection / union,
            1.0 if total == 0 else 2 * intersection / total)

target = np.zeros((10, 10), dtype=bool)
target[2:4, 2:4] = True
all_background = np.zeros_like(target)
partial = target.copy()
partial[3, 3] = False
partial[0, 0] = True
for name, prediction in [("Toàn nền", all_background), ("Tìm một phần", partial)]:
    print(name, "accuracy/IoU/Dice:", [round(v, 4) for v in metrics(prediction, target)])
print("Hai mask rỗng:", metrics(all_background, all_background))
assert metrics(all_background, target) == (0.96, 0.0, 0.0)
assert metrics(partial, target)[1:] == (0.6, 0.75)

Toàn nền accuracy/IoU/Dice: [0.96, 0.0, 0.0]
Tìm một phần accuracy/IoU/Dice: [0.98, 0.6, 0.75]
Hai mask rỗng: (1.0, 1.0, 1.0)


Đoán toàn nền đạt accuracy 0.96, vì 96 trên 100 pixel đúng là nền. Nhưng nó không tìm thấy pixel mực nào, và IoU, Dice đều bằng 0 nói đúng điều đó. Với vật thể nhỏ, accuracy theo pixel gần như vô dụng.

Dự đoán thứ hai có 3 pixel đúng, 1 pixel thừa và 1 pixel sót. IoU là $3/(3+1+1)=0.6$. Dice là $2\cdot3/(4+4)=0.75$. Quy ước cho ảnh rỗng, và cách lấy trung bình qua nhiều ảnh, phải khớp với cách chấm của đề khi làm bài thật.

## Ví dụ 2: Một U-Net nhỏ tìm hình chữ nhật sáng

Mỗi ảnh 8×8 có một hình chữ nhật sáng trên nền tối, cộng chút nhiễu. Nhãn là mask 0 và 1. Bài này rất dễ, chỉ để kiểm các bộ phận của U-Net: nhánh mã hóa, nhánh giải mã, đường nối tắt, và loss nhị phân. Nó chưa phải bài phân đoạn chữ viết thật.

In [ ]:
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

torch.set_num_threads(1)
torch.manual_seed(56)

def make_rectangles(count, seed):
    rng = np.random.default_rng(seed)
    masks = np.zeros((count, 1, 8, 8), dtype=np.float32)
    for i in range(count):
        row, col = rng.integers(1, 5, size=2)
        masks[i, 0, row:row + 3, col:col + 2] = 1
    images = np.clip(0.1 + 0.8 * masks + rng.normal(0, 0.04, masks.shape), 0, 1)
    return torch.tensor(images, dtype=torch.float32), torch.from_numpy(masks)

train_x, train_y = make_rectangles(16, 1)
valid_x, valid_y = make_rectangles(4, 2)
print("Ảnh:", tuple(train_x.shape))
print("Mask đầu:")
print(train_y[0, 0].int().numpy())
print("Pixel foreground mỗi ảnh:", train_y.sum((1, 2, 3)).int().tolist())

Ảnh: (16, 1, 8, 8)
Mask đầu:
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 1 1 0 0 0]
 [0 0 0 1 1 0 0 0]
 [0 0 0 1 1 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]
Pixel foreground mỗi ảnh: [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]


Mỗi mask có 6 pixel vật thể và 58 pixel nền. Đếm như vậy trước khi train giúp bắt sớm những lỗi như nhãn toàn 0, hay vùng bị cắt sai.

Vì vật thể ít hơn nền gần mười lần, ta đặt `pos_weight = 58/6` trong loss. Một pixel vật thể bị đoán sai sẽ bị phạt nặng hơn khoảng 9.67 lần, bù cho chênh lệch số lượng. Đây là một lựa chọn về loss, và phải kiểm bằng metric. Nó không làm thay đổi cách tính IoU.

In [ ]:
class SmallUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encode = nn.Sequential(nn.Conv2d(1, 4, 3, padding=1), nn.ReLU())
        self.middle = nn.Sequential(nn.Conv2d(4, 8, 3, padding=1), nn.ReLU())
        self.decode = nn.Sequential(nn.Conv2d(12, 4, 3, padding=1), nn.ReLU())
        self.head = nn.Conv2d(4, 1, 1)

    def forward(self, x):
        skip = self.encode(x)
        low = self.middle(F.max_pool2d(skip, 2))
        up = F.interpolate(low, size=skip.shape[-2:], mode="nearest")
        return self.head(self.decode(torch.cat([skip, up], dim=1)))

model = SmallUNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
# Hệ số tính từ mask train, không dùng nhãn validation.
positive_weight = (train_y.numel() - train_y.sum()) / train_y.sum()
criterion = nn.BCEWithLogitsLoss(pos_weight=positive_weight.reshape(1))
with torch.no_grad():
    before = criterion(model(valid_x), valid_y).item()
print("Logits batch hai ảnh:", tuple(model(train_x[:2]).shape))
print("pos_weight:", round(positive_weight.item(), 4))
for step in range(80):
    optimizer.zero_grad()
    loss = criterion(model(train_x), train_y)
    loss.backward()
    optimizer.step()
model.eval()
with torch.no_grad():
    logits = model(valid_x)
    after = criterion(logits, valid_y).item()
    prediction = logits.sigmoid() >= 0.5
g = valid_y.bool()
intersection = (prediction & g).sum((1, 2, 3))
union = (prediction | g).sum((1, 2, 3))
iou = intersection.float() / union
print("Valid BCE trước/sau:", round(before, 4), round(after, 4))
print("IoU từng ảnh:", [round(v, 4) for v in iou.tolist()])
print("Ảnh kích thước lẻ:", tuple(model(torch.zeros(1, 1, 9, 7)).shape))
assert after < before and iou.mean().item() > 0.8
assert model(torch.zeros(1, 1, 9, 7)).shape == (1, 1, 9, 7)

Logits batch hai ảnh: (2, 1, 8, 8)
pos_weight: 9.6667
Valid BCE trước/sau: 1.2735 0.0018
IoU từng ảnh: [1.0, 1.0, 1.0, 1.0]
Ảnh kích thước lẻ: (1, 1, 9, 7)


Đi theo hàm `forward`. Nhánh mã hóa giữ ảnh 8×8 với bốn kênh, và kết quả này được giữ lại làm đường nối tắt. Pooling thu xuống 4×4, rồi lớp giữa tạo tám kênh. Phóng to trở lại đúng kích thước của đường nối tắt, rồi ghép hai bên lại: 4 + 8 = 12 kênh. Nhờ phóng to theo đúng kích thước của đường nối tắt, ảnh có kích thước lẻ như 9×7 vẫn ra đúng 9×7.

Sau 80 bước, loss validation giảm từ 1.27 xuống 0.0018, và cả bốn ảnh validation đạt IoU 1. Đó là vì hình chữ nhật sạch quá dễ. Loss làm việc trên logit liên tục. Ngưỡng 0.5 chỉ xuất hiện lúc biến xác suất thành mask để chấm.

## Ví dụ 3: RLE phải đúng thứ tự và đúng điểm bắt đầu

RLE nén mask thành các cặp số: vị trí bắt đầu, và độ dài của một đoạn pixel 1 liên tiếp. Trong ví dụ này, mask được trải phẳng theo từng hàng, và vị trí đếm từ 1. Mask `[[0, 1, 0], [1, 1, 0]]` trải phẳng thành `0 1 0 1 1 0`, nên RLE là `[2, 1, 4, 2]`. Ta kiểm thêm mask rỗng, mask kín, và một lỗi đọc sai thứ tự.

In [ ]:
import numpy as np

def encode(mask):
    mask = np.asarray(mask)
    assert np.isin(mask, [0, 1]).all()
    values = mask.astype(np.uint8).ravel(order="C")
    changes = np.flatnonzero(np.diff(np.r_[0, values, 0])) + 1
    changes[1::2] -= changes[::2]
    return changes.tolist()

def decode(runs, shape):
    assert len(runs) % 2 == 0
    flat = np.zeros(np.prod(shape), dtype=np.uint8)
    end_previous = 0
    for start, length in zip(runs[::2], runs[1::2]):
        assert start > end_previous and length > 0
        assert start - 1 + length <= len(flat)
        flat[start - 1:start - 1 + length] = 1
        end_previous = start - 1 + length
    return flat.reshape(shape, order="C")

mask = np.array([[0, 1, 0], [1, 1, 0]], dtype=np.uint8)
for name, m in [("Mẫu", mask), ("Rỗng", np.zeros_like(mask)), ("Kín", np.ones_like(mask))]:
    runs = encode(m)
    print(name, runs, "round trip:", np.array_equal(m, decode(runs, m.shape)))
assert encode(mask) == [2, 1, 4, 2]
wrong_order = decode(encode(mask), mask.shape).ravel().reshape(mask.shape, order="F")
print("Cùng bytes, đọc theo cột có đúng mask:", np.array_equal(mask, wrong_order))
assert not np.array_equal(mask, wrong_order)

Mẫu [2, 1, 4, 2] round trip: True
Rỗng [] round trip: True
Kín [1, 6] round trip: True
Cùng bytes, đọc theo cột có đúng mask: False


Mask kín cho một đoạn duy nhất `[1, 6]`. Mask rỗng cho danh sách rỗng. Mã hóa rồi giải mã đều trả lại đúng mask ban đầu. Hàm giải mã còn kiểm các đoạn không chồng nhau và không vượt quá mask.

Nhưng đúng chiều đi rồi đi về chỉ chứng tỏ hai hàm của ta khớp với nhau. Dòng cuối đọc cùng dãy số theo thứ tự cột, và ra một mask sai. Có đề trải mask theo hàng, có đề theo cột. Có đề đếm từ 1, có đề từ 0. Trước khi nộp, phải giải mã thử file mẫu của đề bằng hàm của mình, và kiểm mask ra đúng.